<a href="https://colab.research.google.com/github/Jalwaysontop/SatQuery/blob/main/Grounding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# CELL 1 — MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

print("✅ Google Drive mounted")

Mounted at /content/drive
✅ Google Drive mounted


In [2]:
# ============================================================
# CELL 2 — PROJECT PATHS
# ============================================================

import os

DRIVE_ROOT = "/content/drive/MyDrive"

PROJECT_ROOT = os.path.join(
    DRIVE_ROOT,
    "SatQuery_AI"
)

GROUNDING_ROOT = os.path.join(
    PROJECT_ROOT,
    "Grounding"
)

CHECKPOINT_DIR = os.path.join(
    GROUNDING_ROOT,
    "checkpoints"
)

PROCESSED_DIR = os.path.join(
    GROUNDING_ROOT,
    "processed"
)

CONFIG_DIR = os.path.join(
    GROUNDING_ROOT,
    "configs"
)

# ------------------------------------------------------------
# THIS IS THE GOOGLE DRIVE SHORTCUT
# ------------------------------------------------------------

DATASET_SHORTCUT_NAME = "SatQuery_Grounding_Dataset"

DATASET_ROOT = os.path.join(
    DRIVE_ROOT,
    DATASET_SHORTCUT_NAME
)

# Dataset/cache paths through the shortcut
HF_CACHE = os.path.join(
    DATASET_ROOT,
    "hf_cache"
)

DIOR_CACHE = os.path.join(
    DATASET_ROOT,
    "dior_raw"
)

# ------------------------------------------------------------
# Checkpoint paths
# ------------------------------------------------------------

RESUME_STATE = os.path.join(
    CHECKPOINT_DIR,
    "latest_training_state.pt"
)

LATEST_ADAPTER_DIR = os.path.join(
    CHECKPOINT_DIR,
    "latest_adapter"
)

print("=" * 70)
print("PATH CONFIGURATION")
print("=" * 70)

print("Project       :", PROJECT_ROOT)
print("Dataset       :", DATASET_ROOT)
print("HF cache      :", HF_CACHE)
print("DIOR cache    :", DIOR_CACHE)
print("Checkpoints   :", CHECKPOINT_DIR)
print("Processed     :", PROCESSED_DIR)

PATH CONFIGURATION
Project       : /content/drive/MyDrive/SatQuery_AI
Dataset       : /content/drive/MyDrive/SatQuery_Grounding_Dataset
HF cache      : /content/drive/MyDrive/SatQuery_Grounding_Dataset/hf_cache
DIOR cache    : /content/drive/MyDrive/SatQuery_Grounding_Dataset/dior_raw
Checkpoints   : /content/drive/MyDrive/SatQuery_AI/Grounding/checkpoints
Processed     : /content/drive/MyDrive/SatQuery_AI/Grounding/processed


In [6]:
# ============================================================
# CELL 3 — VERIFY DRIVE SHORTCUT
# ============================================================

import os

print("=" * 70)
print("DRIVE SHORTCUT VERIFICATION")
print("=" * 70)

print("Shortcut:")
print(DATASET_ROOT)

print("\nShortcut exists:")
print(os.path.exists(DATASET_ROOT))

assert os.path.exists(
    DATASET_ROOT
), f"""
❌ Dataset shortcut not found.

Expected:
{DATASET_ROOT}

Check the shortcut name in Google Drive.
"""

print("\nShortcut contents:")
for name in os.listdir(DATASET_ROOT)[:30]:
    print(" ", name)

print("\n✅ Dataset shortcut is accessible.")

DRIVE SHORTCUT VERIFICATION
Shortcut:
/content/drive/MyDrive/SatQuery_Grounding_Dataset

Shortcut exists:
True

Shortcut contents:
  dior_raw
  hf_cache

✅ Dataset shortcut is accessible.


In [7]:
# ============================================================
# CELL 4 — VERIFY CHECKPOINT + ANNOTATIONS
# ============================================================

import os
import torch

TRAIN_JSON = os.path.join(
    PROCESSED_DIR,
    "grounding_train.json"
)

VAL_JSON = os.path.join(
    PROCESSED_DIR,
    "grounding_val.json"
)

TEST_JSON = os.path.join(
    PROCESSED_DIR,
    "grounding_test.json"
)

required = {
    "checkpoint":
        RESUME_STATE,

    "latest adapter":
        LATEST_ADAPTER_DIR,

    "train annotations":
        TRAIN_JSON,

    "val annotations":
        VAL_JSON,

    "test annotations":
        TEST_JSON,
}

for name, path in required.items():
    print(
        f"{'✅' if os.path.exists(path) else '❌'} "
        f"{name}: {path}"
    )

assert os.path.isfile(RESUME_STATE)
assert os.path.isdir(LATEST_ADAPTER_DIR)
assert os.path.isfile(TRAIN_JSON)
assert os.path.isfile(VAL_JSON)
assert os.path.isfile(TEST_JSON)

state = torch.load(
    RESUME_STATE,
    map_location="cpu",
    weights_only=False
)

print("\n============================================================")
print("CHECKPOINT INFORMATION")
print("============================================================")

print("Epoch       :", state.get("epoch"))
print("Global step :", state.get("global_step"))
print("Best val    :", state.get("best_val_loss"))

assert (
    state.get("global_step", -1) >= 0
), "Invalid checkpoint global step"

print("\n✅ Checkpoint and annotation files verified.")

✅ checkpoint: /content/drive/MyDrive/SatQuery_AI/Grounding/checkpoints/latest_training_state.pt
✅ latest adapter: /content/drive/MyDrive/SatQuery_AI/Grounding/checkpoints/latest_adapter
✅ train annotations: /content/drive/MyDrive/SatQuery_AI/Grounding/processed/grounding_train.json
✅ val annotations: /content/drive/MyDrive/SatQuery_AI/Grounding/processed/grounding_val.json
✅ test annotations: /content/drive/MyDrive/SatQuery_AI/Grounding/processed/grounding_test.json

CHECKPOINT INFORMATION
Epoch       : 0
Global step : 200
Best val    : inf

✅ Checkpoint and annotation files verified.


In [8]:
# ============================================================
# CELL 5 — INSTALL DEPENDENCIES
# ============================================================

import subprocess
import sys

packages = [
    "torch",
    "torchvision",
    "torchaudio",

    "transformers",
    "accelerate",
    "peft",
    "trl",
    "bitsandbytes",
    "datasets",
    "huggingface_hub",

    "Pillow",
    "matplotlib",
    "tqdm",
    "numpy",
    "scipy",
    "qwen-vl-utils",
]

print("Installing dependencies...")

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    *packages
])

print("✅ Dependencies installed.")

Installing dependencies...
✅ Dependencies installed.


In [9]:
# ============================================================
# CELL 6 — GPU / CUDA CHECK
# ============================================================

import torch

print("=" * 70)
print("GPU ENVIRONMENT")
print("=" * 70)

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())

assert torch.cuda.is_available(), \
    "❌ CUDA GPU is required."

gpu = torch.cuda.get_device_properties(0)

print("GPU     :", gpu.name)
print("VRAM    :", gpu.total_memory / (1024 ** 3), "GB")

device = torch.device("cuda:0")

print("Device  :", device)

GPU ENVIRONMENT
PyTorch : 2.11.0+cu128
CUDA    : True
GPU     : Tesla T4
VRAM    : 14.56317138671875 GB
Device  : cuda:0


In [10]:
# ============================================================
# CELL 7 — LOAD QWEN2.5-VL-3B-INSTRUCT
# ============================================================

import torch

from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig
)

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

print("=" * 70)
print("LOADING QWEN")
print("=" * 70)

print("Model :", MODEL_ID)
print("Cache :", HF_CACHE)

assert os.path.exists(
    HF_CACHE
), f"❌ HF cache not found: {HF_CACHE}"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    cache_dir=HF_CACHE
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    cache_dir=HF_CACHE
)

print("\n✅ Qwen model loaded.")
print("✅ Processor loaded.")

LOADING QWEN
Model : Qwen/Qwen2.5-VL-3B-Instruct
Cache : /content/drive/MyDrive/SatQuery_Grounding_Dataset/hf_cache


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]


✅ Qwen model loaded.
✅ Processor loaded.


In [11]:
# ============================================================
# CELL 8 — LOAD DIOR FROM DRIVE
# ============================================================

from datasets import load_dataset

print("=" * 70)
print("LOADING DIOR")
print("=" * 70)

print("DIOR cache:")
print(DIOR_CACHE)

assert os.path.exists(
    DIOR_CACHE
), f"❌ DIOR cache not found: {DIOR_CACHE}"

dior_ds = load_dataset(
    "HichTala/dior",
    cache_dir=DIOR_CACHE,
    trust_remote_code=True
)

print("\n✅ DIOR loaded.")

print("\nSplits:")
for split in dior_ds.keys():
    print(
        f"{split}: {len(dior_ds[split]):,} rows"
    )

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'HichTala/dior' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'HichTala/dior' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


LOADING DIOR
DIOR cache:
/content/drive/MyDrive/SatQuery_Grounding_Dataset/dior_raw


README.md:   0%|          | 0.00/4.38k [00:00<?, ?B/s]


✅ DIOR loaded.

Splits:
train: 18,000 rows
test: 3,463 rows
validation: 2,000 rows


In [12]:
# ============================================================
# CELL 9 — LOAD GROUNDING ANNOTATIONS
# ============================================================

import json

with open(
    TRAIN_JSON,
    "r",
    encoding="utf-8"
) as f:
    train_annots = json.load(f)

with open(
    VAL_JSON,
    "r",
    encoding="utf-8"
) as f:
    val_annots = json.load(f)

with open(
    TEST_JSON,
    "r",
    encoding="utf-8"
) as f:
    test_annots = json.load(f)

print("Train:", len(train_annots))
print("Val  :", len(val_annots))
print("Test :", len(test_annots))

assert len(train_annots) > 0
assert len(val_annots) > 0
assert len(test_annots) > 0

print("\n✅ Grounding annotations loaded.")

Train: 10880
Val  : 2270
Test : 2366

✅ Grounding annotations loaded.


In [19]:
# ============================================================
# CELL 10 — PREPARE QLoRA BASE MODEL
# ============================================================

import torch

from peft import (
    prepare_model_for_kbit_training,
)

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

LORA_TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
]

print("=" * 70)
print("PREPARING QLoRA BASE MODEL")
print("=" * 70)

# Disable KV cache during training
if hasattr(model, "config"):
    model.config.use_cache = False

# Prepare the 4-bit model for LoRA/QLoRA training
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

# Explicit gradient checkpointing
if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()

print("LoRA rank    :", LORA_R)
print("LoRA alpha   :", LORA_ALPHA)
print("LoRA dropout :", LORA_DROPOUT)
print("Target       :", LORA_TARGET_MODULES)

print("\n✅ QLoRA base model prepared.")

PREPARING QLoRA BASE MODEL
LoRA rank    : 16
LoRA alpha   : 32
LoRA dropout : 0.05
Target       : ['q_proj', 'k_proj', 'v_proj', 'o_proj']

✅ QLoRA base model prepared.


In [20]:
# ============================================================
# CELL 11 — LOAD SAVED CHECKPOINT-200 LORA ADAPTER
# ============================================================

import os
import torch

from peft import PeftModel

print("=" * 70)
print("LOADING SAVED LORA ADAPTER")
print("=" * 70)

print("Adapter path:")
print(LATEST_ADAPTER_DIR)

assert os.path.isdir(
    LATEST_ADAPTER_DIR
), (
    "❌ latest_adapter not found:\n"
    + LATEST_ADAPTER_DIR
)

# Load the adapter saved at the latest checkpoint.
# The adapter_config.json inside this directory contains
# the LoRA configuration used during the original training.
model = PeftModel.from_pretrained(
    model,
    LATEST_ADAPTER_DIR,
    is_trainable=True
)

model.train()

# ------------------------------------------------------------
# Trainable parameters
# ------------------------------------------------------------

trainable_params = [
    p
    for p in model.parameters()
    if p.requires_grad
]

total_trainable = sum(
    p.numel()
    for p in trainable_params
)

print("\nTrainable parameters:", total_trainable)

print("\n✅ Saved LoRA adapter loaded.")
print("✅ Adapter is trainable.")

LOADING SAVED LORA ADAPTER
Adapter path:
/content/drive/MyDrive/SatQuery_AI/Grounding/checkpoints/latest_adapter

Trainable parameters: 7372800

✅ Saved LoRA adapter loaded.
✅ Adapter is trainable.


/usr/local/lib/python3.13/dist-packages/peft/peft_model.py:665: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.base_model.model.base_model.model.model.language_model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.base_model.model.base_model.model.model.language_model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.base_model.model.base_model.model.model.language_model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.base_model.model.base_model.model.model.language_model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.base_model.model.base_model.model.model.language_model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.base_model.model.base_model.model.model.language_model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.base_model.model.base_model.model.model.language_model.layers.0.self_attn.o_proj.lora_A.default.weight', 'bas

In [22]:
# ============================================================
# CELL 12 — GROUNDING DATASET
# ============================================================

import json
import torch

from torch.utils.data import Dataset
from PIL import Image
from qwen_vl_utils import process_vision_info

# ------------------------------------------------------------
# IMPORTANT FOR T4
# ------------------------------------------------------------

MAX_IMAGE_SIZE = 280

SYSTEM_PROMPT = (
    "You are a satellite image analysis assistant. "
    "When asked to locate objects, respond ONLY with bounding box "
    "coordinates in the format: "
    "<boxes>[[x1,y1,x2,y2],...]</boxes> "
    "where coordinates are absolute pixel values "
    "(left,top,right,bottom). "
    "Do not include any other text."
)


class GroundingDataset(Dataset):

    def __init__(
        self,
        json_path,
        dior_ds,
        processor,
        max_img_size=280,
        split="train"
    ):

        self.dior_ds = dior_ds
        self.processor = processor
        self.max_img_size = max_img_size
        self.split = split

        with open(
            json_path,
            "r",
            encoding="utf-8"
        ) as f:
            self.records = json.load(f)

        print(
            f"[{split}] "
            f"{len(self.records):,} examples"
        )

        if len(self.records) == 0:
            raise ValueError(
                f"0 examples in {json_path}"
            )

    def __len__(self):
        return len(self.records)

    def load_image(self, rec):

        dataset_split = rec["dataset_split"]

        row_idx = int(
            rec["row_idx"]
        )

        sample = self.dior_ds[
            dataset_split
        ][row_idx]

        image = sample["image"]

        if image is None:
            raise ValueError(
                f"Image None: {rec['image_id']}"
            )

        if image.mode != "RGB":
            image = image.convert("RGB")

        return image

    def resize_image_and_scale_boxes(
        self,
        image,
        boxes
    ):

        orig_w, orig_h = image.size

        scale = (
            self.max_img_size /
            max(orig_w, orig_h)
        )

        if scale < 1.0:

            new_w = max(
                28,
                int(
                    round(orig_w * scale / 28)
                ) * 28
            )

            new_h = max(
                28,
                int(
                    round(orig_h * scale / 28)
                ) * 28
            )

            # Prevent exceeding max dimension
            max_dim = max(new_w, new_h)

            if max_dim > self.max_img_size:
                adjust = (
                    self.max_img_size /
                    max_dim
                )

                new_w = max(
                    28,
                    int(new_w * adjust / 28) * 28
                )

                new_h = max(
                    28,
                    int(new_h * adjust / 28) * 28
                )

            # Actual scale factors for boxes
            sx = new_w / orig_w
            sy = new_h / orig_h

            image = image.resize(
                (new_w, new_h),
                Image.Resampling.LANCZOS
            )

            scaled_boxes = []

            for x1, y1, x2, y2 in boxes:

                bx1 = max(
                    0,
                    min(
                        new_w,
                        int(round(x1 * sx))
                    )
                )

                by1 = max(
                    0,
                    min(
                        new_h,
                        int(round(y1 * sy))
                    )
                )

                bx2 = max(
                    0,
                    min(
                        new_w,
                        int(round(x2 * sx))
                    )
                )

                by2 = max(
                    0,
                    min(
                        new_h,
                        int(round(y2 * sy))
                    )
                )

                if (
                    bx2 > bx1
                    and by2 > by1
                ):
                    scaled_boxes.append([
                        bx1,
                        by1,
                        bx2,
                        by2
                    ])

        else:

            # Keep original image dimensions but align them
            # to Qwen's 28-pixel vision grid.
            new_w = max(
                28,
                (orig_w // 28) * 28
            )

            new_h = max(
                28,
                (orig_h // 28) * 28
            )

            # If dimensions need adjustment, resize.
            if (
                new_w != orig_w
                or new_h != orig_h
            ):

                sx = new_w / orig_w
                sy = new_h / orig_h

                image = image.resize(
                    (new_w, new_h),
                    Image.Resampling.LANCZOS
                )

                scaled_boxes = []

                for x1, y1, x2, y2 in boxes:

                    bx1 = int(round(x1 * sx))
                    by1 = int(round(y1 * sy))
                    bx2 = int(round(x2 * sx))
                    by2 = int(round(y2 * sy))

                    bx1 = max(
                        0,
                        min(new_w, bx1)
                    )

                    by1 = max(
                        0,
                        min(new_h, by1)
                    )

                    bx2 = max(
                        0,
                        min(new_w, bx2)
                    )

                    by2 = max(
                        0,
                        min(new_h, by2)
                    )

                    if (
                        bx2 > bx1
                        and by2 > by1
                    ):
                        scaled_boxes.append([
                            bx1,
                            by1,
                            bx2,
                            by2
                        ])

            else:

                scaled_boxes = [
                    [
                        int(x1),
                        int(y1),
                        int(x2),
                        int(y2)
                    ]
                    for x1, y1, x2, y2 in boxes
                ]

        return image, scaled_boxes

    def __getitem__(self, idx):

        rec = self.records[idx]

        query = str(
            rec["query"]
        )

        boxes = rec["boxes"]

        # ----------------------------------------------------
        # Load image lazily
        # ----------------------------------------------------

        image = self.load_image(rec)

        # ----------------------------------------------------
        # Resize image + scale boxes
        # ----------------------------------------------------

        image, scaled_boxes = (
            self.resize_image_and_scale_boxes(
                image,
                boxes
            )
        )

        if len(scaled_boxes) == 0:
            raise ValueError(
                f"No valid boxes: "
                f"{rec['image_id']}"
            )

        # ----------------------------------------------------
        # Assistant response
        # ----------------------------------------------------

        response = (
            "<boxes>"
            +
            json.dumps(
                scaled_boxes,
                separators=(",", ":")
            )
            +
            "</boxes>"
        )

        # ----------------------------------------------------
        # Qwen messages
        # ----------------------------------------------------

        messages = [

            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },

            {
                "role": "user",
                "content": [

                    {
                        "type": "image",
                        "image": image
                    },

                    {
                        "type": "text",
                        "text": query
                    }

                ]
            },

            {
                "role": "assistant",
                "content": response
            }
        ]

        # ----------------------------------------------------
        # Chat template
        # ----------------------------------------------------

        text = (
            self.processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )
        )

        # ----------------------------------------------------
        # Vision processing
        # ----------------------------------------------------

        image_inputs, video_inputs = (
            process_vision_info(messages)
        )

        inputs = self.processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            return_tensors="pt",
            padding=False,
            truncation=False
        )

        # ----------------------------------------------------
        # Remove batch dimension
        # ----------------------------------------------------

        input_ids = inputs[
            "input_ids"
        ].squeeze(0).long()

        attention_mask = inputs[
            "attention_mask"
        ].squeeze(0).long()

        pixel_values = inputs.get(
            "pixel_values"
        )

        image_grid_thw = inputs.get(
            "image_grid_thw"
        )

        if image_grid_thw is not None:
            image_grid_thw = (
                image_grid_thw.long()
            )

        # ----------------------------------------------------
        # Assistant response length
        # ----------------------------------------------------

        response_tokens = (
            self.processor.tokenizer(
                response,
                add_special_tokens=False
            )["input_ids"]
        )

        response_len = len(
            response_tokens
        )

        seq_len = input_ids.shape[0]

        if response_len >= seq_len:
            raise ValueError(
                f"Invalid response length: "
                f"{response_len} >= {seq_len}"
            )

        # ----------------------------------------------------
        # Assistant-only labels
        # ----------------------------------------------------

        labels = torch.full_like(
            input_ids,
            fill_value=-100
        )

        response_start = (
            seq_len - response_len
        )

        labels[
            response_start:
        ] = input_ids[
            response_start:
        ]

        if (labels != -100).sum().item() == 0:
          raise ValueError(
              f"No valid labels: "
              f"{rec['image_id']}"
          )

        return {
            "input_ids":
                input_ids,

            "attention_mask":
                attention_mask,

            "labels":
                labels,

            "pixel_values":
                pixel_values,

            "image_grid_thw":
                image_grid_thw
        }


print(
    "✅ GroundingDataset class ready."
)

print(
    "MAX_IMAGE_SIZE =",
    MAX_IMAGE_SIZE
)

✅ GroundingDataset class ready.
MAX_IMAGE_SIZE = 280


In [23]:
# ============================================================
# CELL 13 — CREATE TRAIN / VAL / TEST DATASETS
# ============================================================

train_dataset = GroundingDataset(
    TRAIN_JSON,
    dior_ds,
    processor,
    max_img_size=MAX_IMAGE_SIZE,
    split="train"
)

val_dataset = GroundingDataset(
    VAL_JSON,
    dior_ds,
    processor,
    max_img_size=MAX_IMAGE_SIZE,
    split="val"
)

test_dataset = GroundingDataset(
    TEST_JSON,
    dior_ds,
    processor,
    max_img_size=MAX_IMAGE_SIZE,
    split="test"
)

print("\n" + "=" * 70)
print("DATASETS READY")
print("=" * 70)

print("Train:", len(train_dataset))
print("Val  :", len(val_dataset))
print("Test :", len(test_dataset))

[train] 10,880 examples
[val] 2,270 examples
[test] 2,366 examples

DATASETS READY
Train: 10880
Val  : 2270
Test : 2366


In [24]:
# ============================================================
# CELL 14 — DATALOADERS
# ============================================================

from torch.utils.data import DataLoader

pad_token_id = (
    processor.tokenizer.pad_token_id
)

if pad_token_id is None:
    pad_token_id = (
        processor.tokenizer.eos_token_id
    )

if pad_token_id is None:
    pad_token_id = 0

collator = GroundingDataCollator(
    processor=processor,
    pad_token_id=pad_token_id
)

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    collate_fn=collator,
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=collator,
    num_workers=0,
    pin_memory=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=collator,
    num_workers=0,
    pin_memory=False
)

print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))
print("Test batches :", len(test_loader))

print("\n✅ DataLoaders ready.")

Train batches: 10880
Val batches  : 2270
Test batches : 2366

✅ DataLoaders ready.


In [25]:
# ============================================================
# CELL 15 — SINGLE BATCH SANITY CHECK
# ============================================================

import torch

print("=" * 70)
print("SINGLE BATCH SANITY CHECK")
print("=" * 70)

batch = next(iter(train_loader))

for key, value in batch.items():

    if isinstance(value, torch.Tensor):

        print(
            f"{key:20s}"
            f" shape={tuple(value.shape)}"
            f" dtype={value.dtype}"
        )

    else:

        print(
            f"{key:20s}"
            f" type={type(value)}"
        )

# ------------------------------------------------------------
# Required dtypes
# ------------------------------------------------------------

assert batch["input_ids"].dtype == torch.long

assert (
    batch["attention_mask"].dtype
    == torch.long
)

assert batch["labels"].dtype == torch.long

if "image_grid_thw" in batch:

    assert (
        batch["image_grid_thw"].dtype
        == torch.long
    )

# ------------------------------------------------------------
# Visual information
# ------------------------------------------------------------

if "image_grid_thw" in batch:

    print(
        "\nimage_grid_thw:",
        batch["image_grid_thw"]
    )

print("\n✅ Single batch is valid.")

SINGLE BATCH SANITY CHECK
input_ids            shape=(1, 215) dtype=torch.int64
attention_mask       shape=(1, 215) dtype=torch.int64
labels               shape=(1, 215) dtype=torch.int64
pixel_values         shape=(400, 1176) dtype=torch.float32
image_grid_thw       shape=(1, 3) dtype=torch.int64

image_grid_thw: tensor([[ 1, 20, 20]])

✅ Single batch is valid.


In [26]:
# ============================================================
# CELL 16 — OPTIMIZER + SCHEDULER
# ============================================================

import math

import torch

from torch.optim import AdamW
from transformers import (
    get_cosine_schedule_with_warmup
)

EPOCHS = 3

LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01

GRAD_ACCUMULATION = 8

WARMUP_RATIO = 0.03

CLIP_GRAD_NORM = 1.0

# Save every 100 optimizer steps
SAVE_EVERY_STEPS = 100

LOG_EVERY_STEPS = 10

device = torch.device("cuda:0")

steps_per_epoch = math.ceil(
    len(train_loader) /
    GRAD_ACCUMULATION
)

total_steps = (
    steps_per_epoch *
    EPOCHS
)

warmup_steps = max(
    1,
    int(
        total_steps *
        WARMUP_RATIO
    )
)

optimizer = AdamW(
    trainable_params,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print("=" * 70)
print("TRAINING CONFIG")
print("=" * 70)

print("Epochs          :", EPOCHS)
print("Batch size      :", 1)
print("Grad accumulation:", GRAD_ACCUMULATION)
print("Learning rate   :", LEARNING_RATE)
print("Total steps     :", total_steps)
print("Warmup steps    :", warmup_steps)
print("Save every      :", SAVE_EVERY_STEPS)

TRAINING CONFIG
Epochs          : 3
Batch size      : 1
Grad accumulation: 8
Learning rate   : 0.0002
Total steps     : 4080
Warmup steps    : 122
Save every      : 100


In [29]:
# ============================================================
# CELL 17 — RESTORE CHECKPOINT STATE
# ============================================================

import os
import torch

print("=" * 70)
print("RESTORING CHECKPOINT")
print("=" * 70)

assert os.path.isfile(
    RESUME_STATE
), f"❌ Checkpoint not found:\n{RESUME_STATE}"

# ------------------------------------------------------------
# Load checkpoint
# ------------------------------------------------------------

state = torch.load(
    RESUME_STATE,
    map_location="cpu",
    weights_only=False
)

print("\nCheckpoint keys:")
for key in state.keys():
    print(" ", key)

# ------------------------------------------------------------
# Basic training state
# ------------------------------------------------------------

start_epoch = state.get(
    "epoch",
    0
)

global_step = state.get(
    "global_step",
    0
)

best_val_loss = state.get(
    "best_val_loss",
    float("inf")
)

train_log = state.get(
    "train_log",
    {
        "loss": [],
        "val_loss": [],
        "steps": []
    }
)

print("\nCheckpoint information:")
print("Epoch       :", start_epoch)
print("Global step :", global_step)
print("Best val    :", best_val_loss)

assert global_step > 0, (
    "❌ Invalid checkpoint: global_step <= 0"
)

# ------------------------------------------------------------
# Restore optimizer
# ------------------------------------------------------------

assert "optimizer" in state, (
    "❌ Optimizer state missing from checkpoint."
)

optimizer.load_state_dict(
    state["optimizer"]
)

print("✅ Optimizer state restored")

# ------------------------------------------------------------
# Scheduler
# ------------------------------------------------------------
#
# The checkpoint does NOT contain scheduler state.
#
# DO NOT call scheduler.step() 200 times.
#
# Instead, position the newly-created scheduler directly
# at the saved global step.
# ------------------------------------------------------------

if "scheduler" in state:

    scheduler.load_state_dict(
        state["scheduler"]
    )

    print("✅ Scheduler state restored")

else:

    print(
        "⚠️ Scheduler state is not present."
    )

    # Make sure every optimizer parameter group knows
    # its original learning rate.
    for group in optimizer.param_groups:

        if "initial_lr" not in group:
            group["initial_lr"] = LEARNING_RATE

    # Position scheduler directly at the saved step.
    scheduler.last_epoch = global_step

    # PyTorch LambdaLR internal step counter.
    scheduler._step_count = global_step + 1

    print(
        f"✅ Scheduler positioned at step {global_step}"
    )

# ------------------------------------------------------------
# AMP scaler
# ------------------------------------------------------------

USE_AMP = torch.cuda.is_available()
AMP_DTYPE = torch.float16

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP
)

if "scaler" in state:

    scaler.load_state_dict(
        state["scaler"]
    )

    print("✅ AMP scaler restored")

else:

    print(
        "⚠️ AMP scaler missing. "
        "Using a fresh scaler."
    )

# ------------------------------------------------------------
# Cleanup
# ------------------------------------------------------------

del state

model.train()

print("\n" + "=" * 70)
print("RESUME READY")
print("=" * 70)

print("Epoch       :", start_epoch)
print("Global step :", global_step)
print("Best val    :", best_val_loss)

print(
    "Current LR  :",
    optimizer.param_groups[0]["lr"]
)

print(
    "Scheduler last_epoch:",
    scheduler.last_epoch
)

print(
    "\n✅ Training is ready to continue "
    f"from optimizer step {global_step}."
)

RESTORING CHECKPOINT

Checkpoint keys:
  epoch
  global_step
  best_val_loss
  optimizer
  scaler
  train_log

Checkpoint information:
Epoch       : 0
Global step : 200
Best val    : inf
✅ Optimizer state restored
⚠️ Scheduler state is not present.
✅ Scheduler positioned at step 200
✅ AMP scaler restored

RESUME READY
Epoch       : 0
Global step : 200
Best val    : inf
Current LR  : 0.00019997530458341033
Scheduler last_epoch: 200

✅ Training is ready to continue from optimizer step 200.


In [30]:
# ============================================================
# CELL 18 — RESUME TRAINING FROM CHECKPOINT
# ============================================================
#
# Resume point:
#     global_step = 200
#
# IMPORTANT:
# - Do NOT reset global_step
# - Do NOT recreate optimizer
# - Do NOT call scheduler.step() before optimizer.step()
# - Save every 100 optimizer steps
# - Keep checkpoint-200 safe until a newer checkpoint is saved
# ============================================================

import os
import gc
import time
import json

import torch
from tqdm.auto import tqdm


# ============================================================
# 1. FINAL SAFETY CHECKS
# ============================================================

assert global_step == 200, (
    f"Expected checkpoint step 200, "
    f"but found global_step={global_step}"
)

assert model is not None
assert optimizer is not None
assert scheduler is not None
assert scaler is not None

assert len(train_loader) > 0

print("=" * 70)
print("RESUMING TRAINING")
print("=" * 70)

print("Starting global step :", global_step)
print("Starting epoch       :", start_epoch)
print("Learning rate        :", optimizer.param_groups[0]["lr"])
print("Image size           :", MAX_IMAGE_SIZE)
print("Gradient accumulation:", GRAD_ACCUMULATION)

print(
    "\nCheckpoint policy:"
)
print(
    "  Existing checkpoint : step 200"
)
print(
    "  Next checkpoint     : step 300"
)


# ============================================================
# 2. CHECKPOINT PATHS
# ============================================================

CHECKPOINT_DIR = os.path.join(
    GROUNDING_ROOT,
    "checkpoints"
)

LATEST_ADAPTER_DIR = os.path.join(
    CHECKPOINT_DIR,
    "latest_adapter"
)

RESUME_STATE = os.path.join(
    CHECKPOINT_DIR,
    "latest_training_state.pt"
)

LOG_DIR = os.path.join(
    GROUNDING_ROOT,
    "logs"
)

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

os.makedirs(
    LOG_DIR,
    exist_ok=True
)

LOG_FILE = os.path.join(
    LOG_DIR,
    "training_log.json"
)


# ============================================================
# 3. SAVE CHECKPOINT FUNCTION
# ============================================================

def save_training_checkpoint(
    epoch,
    global_step,
    best_val_loss,
    train_log
):

    print(
        f"\n{'=' * 70}"
    )

    print(
        f"SAVING CHECKPOINT — STEP {global_step}"
    )

    print(
        f"{'=' * 70}"
    )

    # --------------------------------------------------------
    # Save adapter into temporary directory first
    # --------------------------------------------------------

    temp_dir = (
        LATEST_ADAPTER_DIR +
        "_tmp"
    )

    if os.path.exists(temp_dir):

        import shutil

        shutil.rmtree(
            temp_dir
        )

    model.save_pretrained(
        temp_dir,
        safe_serialization=True
    )

    processor.save_pretrained(
        temp_dir
    )

    # --------------------------------------------------------
    # Replace latest adapter only after successful save
    # --------------------------------------------------------

    if os.path.exists(
        LATEST_ADAPTER_DIR
    ):

        import shutil

        shutil.rmtree(
            LATEST_ADAPTER_DIR
        )

    os.rename(
        temp_dir,
        LATEST_ADAPTER_DIR
    )

    # --------------------------------------------------------
    # Save training state
    # --------------------------------------------------------
    #
    # Note:
    # This checkpoint format intentionally records the
    # optimizer/scaler. The scheduler is reconstructed from
    # global_step when needed.
    # --------------------------------------------------------

    checkpoint_state = {

        "epoch":
            epoch,

        "global_step":
            global_step,

        "best_val_loss":
            best_val_loss,

        "optimizer":
            optimizer.state_dict(),

        "scaler":
            scaler.state_dict(),

        "train_log":
            train_log,
    }

    torch.save(
        checkpoint_state,
        RESUME_STATE
    )

    # --------------------------------------------------------
    # Save readable training log
    # --------------------------------------------------------

    with open(
        LOG_FILE,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            train_log,
            f,
            indent=2
        )

    print(
        f"✅ Checkpoint saved at step {global_step}"
    )

    print(
        f"   Adapter : {LATEST_ADAPTER_DIR}"
    )

    print(
        f"   State   : {RESUME_STATE}"
    )

    return True


# ============================================================
# 4. TRAINING LOOP
# ============================================================

model.train()

for epoch in range(
    start_epoch,
    EPOCHS
):

    print(
        f"\n{'=' * 70}"
    )

    print(
        f"EPOCH {epoch + 1}/{EPOCHS}"
    )

    print(
        f"{'=' * 70}"
    )

    epoch_start = time.time()

    optimizer.zero_grad(
        set_to_none=True
    )

    accum_count = 0
    accum_loss = 0.0

    pbar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}",
        leave=True
    )

    for batch_idx, batch in enumerate(pbar):

        # ----------------------------------------------------
        # Move tensors to CUDA
        # ----------------------------------------------------

        try:

            batch = {
                k: (
                    v.to(
                        device,
                        non_blocking=True
                    )
                    if isinstance(
                        v,
                        torch.Tensor
                    )
                    else v
                )
                for k, v in batch.items()
            }

        except RuntimeError as e:

            if "out of memory" in str(e).lower():

                print(
                    "\n⚠️ CUDA OOM while moving batch."
                )

                optimizer.zero_grad(
                    set_to_none=True
                )

                del batch

                gc.collect()

                torch.cuda.empty_cache()

                continue

            raise


        # ----------------------------------------------------
        # Forward + loss
        # ----------------------------------------------------

        try:

            with torch.autocast(
                device_type="cuda",
                dtype=AMP_DTYPE,
                enabled=USE_AMP
            ):

                outputs = model(
                    input_ids=batch[
                        "input_ids"
                    ],

                    attention_mask=batch[
                        "attention_mask"
                    ],

                    pixel_values=batch.get(
                        "pixel_values"
                    ),

                    image_grid_thw=batch.get(
                        "image_grid_thw"
                    ),

                    labels=batch[
                        "labels"
                    ]
                )

                raw_loss = outputs.loss

                loss = (
                    raw_loss /
                    GRAD_ACCUMULATION
                )

            # ------------------------------------------------
            # Non-finite check
            # ------------------------------------------------

            if not torch.isfinite(
                raw_loss
            ):

                print(
                    f"\n⚠️ Non-finite loss "
                    f"at batch {batch_idx}"
                )

                optimizer.zero_grad(
                    set_to_none=True
                )

                del outputs
                del raw_loss
                del loss
                del batch

                gc.collect()
                torch.cuda.empty_cache()

                continue

            # ------------------------------------------------
            # Backward
            # ------------------------------------------------

            scaler.scale(
                loss
            ).backward()

            accum_count += 1

            accum_loss += (
                raw_loss.item()
            )

        except RuntimeError as e:

            if "out of memory" in str(e).lower():

                print(
                    "\n⚠️ CUDA OOM during "
                    "forward/backward."
                )

                print(
                    "Skipping this batch."
                )

                optimizer.zero_grad(
                    set_to_none=True
                )

                if "outputs" in locals():
                    del outputs

                if "raw_loss" in locals():
                    del raw_loss

                if "loss" in locals():
                    del loss

                del batch

                gc.collect()

                torch.cuda.empty_cache()

                continue

            raise


        # ----------------------------------------------------
        # Optimizer step
        # ----------------------------------------------------

        is_accumulation_complete = (
            accum_count >=
            GRAD_ACCUMULATION
        )

        is_last_batch = (
            batch_idx ==
            len(train_loader) - 1
        )

        if (
            is_accumulation_complete
            or is_last_batch
        ):

            # ------------------------------------------------
            # Unscale
            # ------------------------------------------------

            scaler.unscale_(
                optimizer
            )

            # ------------------------------------------------
            # Gradient clipping
            # ------------------------------------------------

            torch.nn.utils.clip_grad_norm_(
                trainable_params,
                CLIP_GRAD_NORM
            )

            # ------------------------------------------------
            # Optimizer update
            # ------------------------------------------------

            scaler.step(
                optimizer
            )

            scaler.update()

            # ------------------------------------------------
            # Scheduler update
            #
            # This is the FIRST scheduler.step() after
            # optimizer.step(), so no PyTorch ordering warning.
            # ------------------------------------------------

            scheduler.step()

            optimizer.zero_grad(
                set_to_none=True
            )

            # ------------------------------------------------
            # Global optimizer step
            # ------------------------------------------------

            global_step += 1

            avg_loss = (
                accum_loss /
                max(accum_count, 1)
            )

            accum_loss = 0.0
            accum_count = 0

            # ------------------------------------------------
            # Progress
            # ------------------------------------------------

            current_lr = (
                optimizer.param_groups[0]["lr"]
            )

            pbar.set_postfix(
                {
                    "loss":
                        f"{avg_loss:.4f}",

                    "lr":
                        f"{current_lr:.2e}",

                    "step":
                        global_step
                }
            )

            # ------------------------------------------------
            # Logging
            # ------------------------------------------------

            if (
                global_step %
                LOG_EVERY_STEPS
                == 0
            ):

                train_log.setdefault(
                    "steps",
                    []
                )

                train_log.setdefault(
                    "loss",
                    []
                )

                train_log["steps"].append(
                    global_step
                )

                train_log["loss"].append(
                    avg_loss
                )

                print(
                    f"\nStep {global_step} | "
                    f"Loss {avg_loss:.4f} | "
                    f"LR {current_lr:.3e}"
                )

            # ------------------------------------------------
            # Checkpoint every 100 optimizer steps
            #
            # 300, 400, 500...
            # ------------------------------------------------

            if (
                global_step %
                SAVE_EVERY_STEPS
                == 0
            ):

                save_training_checkpoint(
                    epoch=epoch,
                    global_step=global_step,
                    best_val_loss=best_val_loss,
                    train_log=train_log
                )

        # ----------------------------------------------------
        # Cleanup
        # ----------------------------------------------------

        del outputs
        del raw_loss
        del loss
        del batch

    # ========================================================
    # END OF EPOCH
    # ========================================================

    epoch_time = (
        time.time() -
        epoch_start
    )

    print(
        f"\nEpoch {epoch + 1} finished "
        f"in {epoch_time / 60:.2f} minutes."
    )


# ============================================================
# FINAL SAVE
# ============================================================

if global_step > 0:

    save_training_checkpoint(
        epoch=EPOCHS,
        global_step=global_step,
        best_val_loss=best_val_loss,
        train_log=train_log
    )


# ============================================================
# DONE
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "TRAINING LOOP FINISHED"
)

print(
    "=" * 70
)

print(
    "Final global step:",
    global_step
)

print(
    "Latest checkpoint:",
    RESUME_STATE
)

RESUMING TRAINING
Starting global step : 200
Starting epoch       : 0
Learning rate        : 0.00019997530458341033
Image size           : 280
Gradient accumulation: 8

Checkpoint policy:
  Existing checkpoint : step 200
  Next checkpoint     : step 300

EPOCH 1/3


Epoch 1/3:   0%|          | 0/10880 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.



Step 210 | Loss 0.8582 | LR 1.998e-04

Step 220 | Loss 0.9205 | LR 1.997e-04

Step 230 | Loss 0.8132 | LR 1.996e-04

Step 240 | Loss 0.9922 | LR 1.996e-04

Step 250 | Loss 0.8509 | LR 1.995e-04

Step 260 | Loss 0.7871 | LR 1.994e-04

Step 270 | Loss 0.8573 | LR 1.993e-04

Step 280 | Loss 0.7082 | LR 1.992e-04


ValueError: No valid boxes: 4931

In [31]:
# ============================================================
# PATCH — FIX TINY BOXES AFTER 280px RESIZING
# ============================================================

import math
from PIL import Image

def safe_resize_image_and_scale_boxes(
    self,
    image,
    boxes
):
    orig_w, orig_h = image.size

    # Keep the same 280px maximum resolution.
    scale = (
        self.max_img_size /
        max(orig_w, orig_h)
    )

    if scale < 1.0:

        new_w = max(
            1,
            int(round(orig_w * scale))
        )

        new_h = max(
            1,
            int(round(orig_h * scale))
        )

        image = image.resize(
            (new_w, new_h),
            Image.Resampling.LANCZOS
        )

        sx = new_w / orig_w
        sy = new_h / orig_h

    else:

        new_w = orig_w
        new_h = orig_h

        sx = 1.0
        sy = 1.0

    scaled_boxes = []

    for box in boxes:

        if len(box) != 4:
            continue

        x1, y1, x2, y2 = box

        # Scale coordinates
        bx1 = int(round(x1 * sx))
        by1 = int(round(y1 * sy))
        bx2 = int(round(x2 * sx))
        by2 = int(round(y2 * sy))

        # Clamp
        bx1 = max(
            0,
            min(new_w, bx1)
        )

        by1 = max(
            0,
            min(new_h, by1)
        )

        bx2 = max(
            0,
            min(new_w, bx2)
        )

        by2 = max(
            0,
            min(new_h, by2)
        )

        # ----------------------------------------------------
        # Preserve tiny but valid boxes.
        # ----------------------------------------------------

        if x2 > x1 and y2 > y1:

            # Make sure the box has at least 1 pixel width.
            if bx2 <= bx1:
                bx2 = min(
                    new_w,
                    bx1 + 1
                )

            # Make sure the box has at least 1 pixel height.
            if by2 <= by1:
                by2 = min(
                    new_h,
                    by1 + 1
                )

            # If the box was at the extreme boundary,
            # shift the starting coordinate back by 1 pixel.
            if bx2 <= bx1:
                bx1 = max(
                    0,
                    bx2 - 1
                )

            if by2 <= by1:
                by1 = max(
                    0,
                    by2 - 1
                )

            if (
                bx2 > bx1
                and by2 > by1
            ):
                scaled_boxes.append([
                    bx1,
                    by1,
                    bx2,
                    by2
                ])

    return image, scaled_boxes


# ------------------------------------------------------------
# Monkey-patch the CURRENT dataset class.
# Existing dataset objects will use the new method.
# ------------------------------------------------------------

GroundingDataset.resize_image_and_scale_boxes = (
    safe_resize_image_and_scale_boxes
)

print(
    "✅ GroundingDataset resize method patched."
)

✅ GroundingDataset resize method patched.


In [32]:
# ============================================================
# TEST IMAGE 4931
# ============================================================

# Find the record containing image_id 4931.
bad_indices = [
    i
    for i, rec in enumerate(train_dataset.records)
    if str(rec.get("image_id")) == "4931"
]

print("Matching records:", bad_indices)

assert len(bad_indices) > 0, (
    "Could not find image 4931 in train dataset."
)

idx = bad_indices[0]

print("Dataset index:", idx)
print("Record:", train_dataset.records[idx])

# Test complete __getitem__
sample = train_dataset[idx]

print("\n✅ Image 4931 can now be processed.")

for key, value in sample.items():

    if hasattr(value, "shape"):
        print(
            key,
            tuple(value.shape),
            value.dtype
            if hasattr(value, "dtype")
            else ""
        )
    else:
        print(
            key,
            type(value)
        )

Matching records: [6304, 6305, 6306, 6307, 6308, 6309]
Dataset index: 6304
Record: {'image_id': '4931', 'dataset_split': 'train', 'row_idx': 2930, 'query': 'Show me all baseballfields.', 'boxes': [[597, 612, 621, 636], [614, 645, 632, 661], [638, 641, 659, 657]], 'type': 'multi', 'img_w': 800, 'img_h': 800}

✅ Image 4931 can now be processed.
input_ids (239,) torch.int64
attention_mask (239,) torch.int64
labels (239,) torch.int64
pixel_values (400, 1176) torch.float32
image_grid_thw (1, 3) torch.int64


In [33]:
# ============================================================
# TEST DATALOADER AFTER PATCH
# ============================================================

print("Testing DataLoader...")

test_batch = None

for i, b in enumerate(train_loader):

    test_batch = b

    print(
        f"✅ Successfully loaded batch {i}"
    )

    if i >= 3:
        break

del test_batch

print("\n✅ DataLoader passed the first 4 tested batches.")

Testing DataLoader...
✅ Successfully loaded batch 0
✅ Successfully loaded batch 1
✅ Successfully loaded batch 2
✅ Successfully loaded batch 3

✅ DataLoader passed the first 4 tested batches.


In [34]:
print("Current global_step:", global_step)
print("Current LR:", optimizer.param_groups[0]["lr"])
print("Scheduler last_epoch:", scheduler.last_epoch)

Current global_step: 286
Current LR: 0.0001991539568227664
Scheduler last_epoch: 286


In [ ]:
# ============================================================
# CELL 19 — CONTINUE FROM CURRENT IN-MEMORY STEP
# ============================================================

import gc
import time
import os
import json

import torch
from tqdm.auto import tqdm

print("=" * 70)
print("CONTINUING CURRENT TRAINING")
print("=" * 70)

print("Current global step :", global_step)
print("Current epoch       :", start_epoch)
print("Current LR          :", optimizer.param_groups[0]["lr"])

assert global_step >= 286, (
    f"Unexpected global_step={global_step}"
)

model.train()

# ------------------------------------------------------------
# Continue the current epoch.
#
# We intentionally start a fresh shuffled pass through the
# DataLoader because the exact DataLoader position at step 286
# is not stored in the checkpoint.
# ------------------------------------------------------------

optimizer.zero_grad(
    set_to_none=True
)

accum_count = 0
accum_loss = 0.0

pbar = tqdm(
    train_loader,
    desc=f"Continue training | epoch {start_epoch + 1}/{EPOCHS}",
    leave=True
)

for batch_idx, batch in enumerate(pbar):

    # --------------------------------------------------------
    # Move batch to GPU
    # --------------------------------------------------------

    batch = {
        k: (
            v.to(
                device,
                non_blocking=True
            )
            if isinstance(v, torch.Tensor)
            else v
        )
        for k, v in batch.items()
    }

    # --------------------------------------------------------
    # Forward / backward
    # --------------------------------------------------------

    try:

        with torch.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE,
            enabled=USE_AMP
        ):

            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                pixel_values=batch.get("pixel_values"),
                image_grid_thw=batch.get("image_grid_thw"),
                labels=batch["labels"]
            )

            raw_loss = outputs.loss

            loss = (
                raw_loss /
                GRAD_ACCUMULATION
            )

        if not torch.isfinite(raw_loss):

            print(
                f"\n⚠️ Non-finite loss at batch {batch_idx}. "
                "Skipping."
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            del outputs
            del raw_loss
            del loss
            del batch

            continue

        scaler.scale(loss).backward()

        accum_count += 1
        accum_loss += raw_loss.item()

    except RuntimeError as e:

        if "out of memory" in str(e).lower():

            print(
                "\n⚠️ CUDA OOM. Skipping batch."
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            gc.collect()
            torch.cuda.empty_cache()

            continue

        raise

    # --------------------------------------------------------
    # Optimizer update
    # --------------------------------------------------------

    is_accumulation_complete = (
        accum_count >= GRAD_ACCUMULATION
    )

    is_last_batch = (
        batch_idx ==
        len(train_loader) - 1
    )

    if (
        is_accumulation_complete
        or is_last_batch
    ):

        scaler.unscale_(
            optimizer
        )

        torch.nn.utils.clip_grad_norm_(
            trainable_params,
            CLIP_GRAD_NORM
        )

        scaler.step(
            optimizer
        )

        scaler.update()

        # IMPORTANT:
        # optimizer.step() comes before scheduler.step()
        scheduler.step()

        optimizer.zero_grad(
            set_to_none=True
        )

        global_step += 1

        avg_loss = (
            accum_loss /
            max(accum_count, 1)
        )

        accum_loss = 0.0
        accum_count = 0

        current_lr = (
            optimizer.param_groups[0]["lr"]
        )

        pbar.set_postfix(
            {
                "loss":
                    f"{avg_loss:.4f}",
                "lr":
                    f"{current_lr:.2e}",
                "step":
                    global_step
            }
        )

        if global_step % 10 == 0:

            print(
                f"\nStep {global_step} | "
                f"Loss {avg_loss:.4f} | "
                f"LR {current_lr:.3e}"
            )

        # ----------------------------------------------------
        # SAVE EVERY 100 STEPS
        # ----------------------------------------------------

        if global_step % SAVE_EVERY_STEPS == 0:

            print(
                f"\n{'=' * 70}"
            )

            print(
                f"SAVING CHECKPOINT — STEP {global_step}"
            )

            print(
                f"{'=' * 70}"
            )

            temp_dir = (
                LATEST_ADAPTER_DIR +
                "_tmp"
            )

            if os.path.exists(temp_dir):

                import shutil

                shutil.rmtree(
                    temp_dir
                )

            model.save_pretrained(
                temp_dir,
                safe_serialization=True
            )

            processor.save_pretrained(
                temp_dir
            )

            if os.path.exists(
                LATEST_ADAPTER_DIR
            ):

                import shutil

                shutil.rmtree(
                    LATEST_ADAPTER_DIR
                )

            os.rename(
                temp_dir,
                LATEST_ADAPTER_DIR
            )

            torch.save(
                {
                    "epoch":
                        start_epoch,

                    "global_step":
                        global_step,

                    "best_val_loss":
                        best_val_loss,

                    "optimizer":
                        optimizer.state_dict(),

                    "scheduler":
                        scheduler.state_dict(),

                    "scaler":
                        scaler.state_dict(),

                    "train_log":
                        train_log,
                },
                RESUME_STATE
            )

            with open(
                LOG_FILE,
                "w",
                encoding="utf-8"
            ) as f:

                json.dump(
                    train_log,
                    f,
                    indent=2
                )

            print(
                f"✅ CHECKPOINT {global_step} SAVED"
            )

    # --------------------------------------------------------
    # Cleanup
    # --------------------------------------------------------

    del outputs
    del raw_loss
    del loss
    del batch

print(
    "\n" + "=" * 70
)

print(
    "CONTINUATION LOOP FINISHED"
)

print(
    "=" * 70
)

print(
    "Final global step:",
    global_step
)

CONTINUING CURRENT TRAINING
Current global step : 286
Current epoch       : 0
Current LR          : 0.0001991539568227664


Continue training | epoch 1/3:   0%|          | 0/10880 [00:00<?, ?it/s]


Step 290 | Loss 0.7731 | LR 1.991e-04

Step 300 | Loss 0.6976 | LR 1.990e-04

SAVING CHECKPOINT — STEP 300
✅ CHECKPOINT 300 SAVED

Step 310 | Loss 0.8421 | LR 1.989e-04

Step 320 | Loss 0.7808 | LR 1.988e-04

Step 330 | Loss 0.6521 | LR 1.986e-04

Step 340 | Loss 0.7481 | LR 1.985e-04

⚠️ CUDA OOM. Skipping batch.

Step 350 | Loss 0.7441 | LR 1.984e-04

Step 360 | Loss 0.7950 | LR 1.982e-04

Step 370 | Loss 0.8726 | LR 1.981e-04

Step 380 | Loss 0.7594 | LR 1.979e-04

Step 390 | Loss 0.6874 | LR 1.977e-04

Step 400 | Loss 0.7526 | LR 1.976e-04

SAVING CHECKPOINT — STEP 400
✅ CHECKPOINT 400 SAVED

Step 410 | Loss 0.7897 | LR 1.974e-04

Step 420 | Loss 0.8145 | LR 1.972e-04

Step 430 | Loss 0.8432 | LR 1.970e-04

Step 440 | Loss 0.7528 | LR 1.968e-04

Step 450 | Loss 0.7460 | LR 1.966e-04

Step 460 | Loss 0.7309 | LR 1.964e-04

Step 470 | Loss 0.7448 | LR 1.962e-04

Step 480 | Loss 0.7223 | LR 1.960e-04

Step 490 | Loss 0.7315 | LR 1.958e-04

Step 500 | Loss 0.6576 | LR 1.955e-04

SAVIN